In [1]:
import pandas as pd

def clean_table():
    """
    Remove rows from table_subscribed.csv that appear in table_bounced.csv
    based on email addresses, and save the clean table to table_clean.csv
    """
    
    # Read the main subscription table
    print("Reading table_subscribed.csv...")
    try:
        # Try with tab separator first, then fall back to comma
        try:
            subscribed_df = pd.read_csv('table_subscribed.csv', sep='\t')
        except:
            subscribed_df = pd.read_csv('Jan20-2026_Subscribed.csv') #exact name here
    except FileNotFoundError:
        print("Error: table_subscribed.csv not found in current directory")
        return
    except Exception as e:
        print(f"Error reading table_subscribed.csv: {e}")
        return
    
    # Read the bounce table
    print("Reading table_bounced.csv...")
    try:
        # Try with tab separator first, then fall back to comma
        try:
            bounced_df = pd.read_csv('table_bounced.csv', sep='\t')
        except:
            bounced_df = pd.read_csv('Jan20-2026_Bounce.csv') #exact name here
    except FileNotFoundError:
        print("Error: table_bounced.csv not found in current directory")
        return
    except Exception as e:
        print(f"Error reading table_bounced.csv: {e}")
        return
    
    # Display initial counts
    print(f"\nInitial count in subscribed table: {len(subscribed_df)}")
    print(f"Count in bounced table: {len(bounced_df)}")
    
    # Check column names
    print(f"\nColumns in subscribed table: {list(subscribed_df.columns)}")
    print(f"Columns in bounced table: {list(bounced_df.columns)}")
    
    # Find the email column in bounced table (could be 'Email' or 'Email Address')
    email_col_bounced = None
    for col in bounced_df.columns:
        if 'email' in col.lower():
            email_col_bounced = col
            break
    
    # Find the email column in subscribed table
    email_col_subscribed = None
    for col in subscribed_df.columns:
        if 'email' in col.lower():
            email_col_subscribed = col
            break
    
    if not email_col_bounced or not email_col_subscribed:
        print("\nError: Could not find email columns in one or both tables")
        return
    
    print(f"\nUsing column '{email_col_subscribed}' as email in subscribed table")
    print(f"Using column '{email_col_bounced}' as email in bounced table")
    
    # Extract email addresses from bounced table
    bounced_emails = bounced_df[email_col_bounced].dropna().str.lower().tolist()
    
    # Filter out rows from subscribed table where email is in bounced list
    clean_df = subscribed_df[~subscribed_df[email_col_subscribed].str.lower().isin(bounced_emails)]
    
    # Display results
    removed_count = len(subscribed_df) - len(clean_df)
    print(f"\nRemoved {removed_count} rows")
    print(f"Final count in clean table: {len(clean_df)}")
    
    # Save the clean table
    print("\nSaving clean table to table_clean.csv...")
    try:
        clean_df.to_csv('table_clean.csv', index=False)
        print("Successfully saved table_clean.csv")
        
        # Show a few examples of removed emails
        removed_df = subscribed_df[subscribed_df[email_col_subscribed].str.lower().isin(bounced_emails)]
        removed_emails = removed_df[email_col_subscribed].tolist()
        if removed_emails:
            print(f"\nExamples of removed emails:")
            for email in removed_emails[:5]:  # Show first 5
                print(f"  - {email}")
            if len(removed_emails) > 5:
                print(f"  ... and {len(removed_emails) - 5} more")
    except Exception as e:
        print(f"Error saving table_clean.csv: {e}")

if __name__ == "__main__":
    clean_table()

Reading table_subscribed.csv...
Reading table_bounced.csv...

Initial count in subscribed table: 1483
Count in bounced table: 212

Columns in subscribed table: ['Email Address', 'First Name', 'Last Name', 'Address', 'Phone Number', 'Birthday', 'Company', 'Role', 'District', 'State', 'MEMBER_RATING', 'OPTIN_TIME', 'OPTIN_IP', 'CONFIRM_TIME', 'CONFIRM_IP', 'GMTOFF', 'DSTOFF', 'TIMEZONE', 'CC', 'REGION', 'LAST_CHANGED', 'LEID', 'EUID', 'NOTES', 'TAGS']
Columns in bounced table: ['Email', 'Bounce type', 'Bounce reason', 'First Name', 'Last Name', 'Address', 'Phone Number', 'Birthday', 'Company', 'Role', 'District', 'State', 'Rating', 'Last changed']

Using column 'Email Address' as email in subscribed table
Using column 'Email' as email in bounced table

Removed 191 rows
Final count in clean table: 1292

Saving clean table to table_clean.csv...
Successfully saved table_clean.csv

Examples of removed emails:
  - carlos.sanchez@cvesd.org
  - jbowman@nsd.org
  - williams.kevin@bsd405.org
  - 